<div class="row">
  <div class="column">
    <img src="./img/logo-onera.png" width="200">
  </div>
  <div class="column">
    <img src="./img/logo-ISAE_SUPAERO.png" width="200">
  </div>
</div>

# Mission with HE power train

The following Notebook shows the user an example of how to use the newly developed power train builder along with the mission from FAST-OAD_CS23.

In [ ]:
import os.path as pth
from shutil import rmtree
import logging

import pytest

import openmdao.api as om
import fastoad.api as oad
import fastga.utils.postprocessing.analysis_and_plots as api_plots

from fastga.utils.postprocessing.analysis_and_plots import (
    mass_breakdown_bar_plot,
    aircraft_geometry_plot,
)

from utils.filter_residuals import filter_residuals

from fastga_he.gui.power_train_network_viewer import power_train_network_viewer
from fastga_he.gui.power_train_weight_breakdown import power_train_mass_breakdown

#new
import pandas as pd
from bs4 import BeautifulSoup
import xml.etree.cElementTree as et

#Densidades de bateria:

densities = [300.0] #[300.0, 350.0, 400.0]

#Dados de range de estudo

distances = [400.0] #[100.0, 150.0, 200.0]

#Baselines para comaparação:

base_100 = ["-","-","-" ,255.05,3.07,972.80,0,2378.74,1140.00,2270.00,"-","-","-","-","-","-","-","-","-"]

base_150 = ["-","-","-" ,349.60,2.53,1333.40,0,2378.74,1140.00,2270.00,"-","-","-","-","-","-","-","-","-"]

base_200 = ["-","-","-" ,590.13,2.13,2250.80,0,2378.74,1140.00,2270.00,"-","-","-","-","-","-","-","-","-"]

base_250 = ["-","-","-" ,"-","-","-","-","-","-","-","-","-","-","-","-","-","-","-","-"]

base_300 = ["-","-","-" ,"-","-","-","-","-","-","-","-","-","-","-","-","-","-","-","-"]

base_350 = ["-","-","-" ,"-","-","-","-","-","-","-","-","-","-","-","-","-","-","-","-"]

base_400 = ["-","-","-" ,"-","-","-","-","-","-","-","-","-","-","-","-","-","-","-","-"]

base_450 = ["-","-","-" ,"-","-","-","-","-","-","-","-","-","-","-","-","-","-","-","-"]

base_500 = ["-","-","-" ,"-","-","-","-","-","-","-","-","-","-","-","-","-","-","-","-"]

base_550 = ["-","-","-" ,"-","-","-","-","-","-","-","-","-","-","-","-","-","-","-","-"]

baseline = {400: base_100} #na real que é 150, 250 e 500 baseline = {100: base_100,150: base_150, 200: base_200}

#Taxa de hibridização

n = 6 #numero de motores eletricos. MODIFICAR COM O NUMERO DE MOTORES

hibrid = [0.0] #para 4 motores deu pau com 0.25. Testar com 0.2 pra ver até onde ele vai e dps retestar com 0.25 pra tentar descobrir o erro. [0.0, 0.05, 0.1, 0.15, 0.20, 0.25]

#Rodou tudo com He = 0.2. Vamo testar para 2 motores

#Para 2 motores, deu pau com He = 0.15.

#Rodou com He = 0.1

"""16/02/2026"""
#Engine sizing data [rpm, voltage]
EMRAX = {'EMRAX_188': [8000, 660],
         'EMRAX_208': [7000, 690],
         'EMRAX_228': [6500, 830],
         'EMRAX_268': [4500, 830],
         'EMRAX_348': [3250, 830]}

#Criação do DataFrame para os dados

df = pd.DataFrame()

DATA_FOLDER_PATH =  "data"
RESULTS_FOLDER_PATH = "results"

for engine in EMRAX: #nao vai dar certo. tem que mudar a variação no dicionario para pegar bem os valores dos motores
    """17/02/2026"""
    #Modificação de posição para compreender o dimensionamento de motores

    #Modificar Input_CaravanDEP ao inves de oad_process_inputs_CaravanDEP_retrofit → isso sera feito automatico com a mudança de Input_CaravanDEP.   
    with open(r".\data\Input_CaravanDEP.xml", "r", encoding="utf-8") as f: #MODIFICAR COM O NUMERO DE MOTORES
        soup = BeautifulSoup(f, "xml")  # use "xml" parser for XML mode
    
    PMSM_tag = soup.find("data/propulsion/he_power_train/PMSM")
    

    for i in range(n):
        motor_i_tag = PMSM_tag.find("./motor_" + str(i+1))
        #change value of RPM and Voltage
        motor_i_tag.find('rpm_rating').string = str(EMRAX[engine][0])
        motor_i_tag.find("voltage_caliber").string = str(EMRAX[engine][1])
    
    
    for rho in densities:
        for dist in distances:

            datum = {"Engine":[engine],"rho_bat": [baseline[dist][0]],"range_nmi": ["Base" + " " + str(dist)], "he": [baseline[dist][2]] , "fuel_mass": [baseline[dist][3]], "emission_factor": [baseline[dist][4]], "total_emissions": [baseline[dist][5]] ,"red_emission": [baseline[dist][6]], "OWE": [baseline[dist][7]], "payload": [baseline[dist][8]], "empty_mass": [baseline[dist][9]], "he_mass": [baseline[dist][10]], "dc_dc_mass": [baseline[dist][11]], "dc_bus_mass": [baseline[dist][12]], "dc_cable_harn": [baseline[dist][13]], "motor_mass": [baseline[dist][14]], "battery_mass": [baseline[dist][15]], "fuel_flowed": [baseline[dist][16]], "inverter_mass": [baseline[dist][17]], "turboshaft_mass": [baseline[dist][18]]}

            for he in hibrid:

                range_tag = soup.find("TLAR").find("range")
                print("Old value:", range_tag.text, range_tag.get("units"))

                # Modify the value and the units
                range_tag.string = str(dist)
                range_tag["units"] = "nmi"

                # Modification for thrust vector

                thrust_tag = soup.find('thrust_distribution')

                thrust_part = float(he/n/(1.0 - he))

                thrust_tag.string = str([1.0, thrust_part, thrust_part, thrust_part, thrust_part, thrust_part, thrust_part]) #MODIFICARCOM O NUMERO DE MOTORES N thrust_parts

            # 2 motores str([1.0, thrust_part, thrust_part])
            # 4 motores str([1.0, thrust_part, thrust_part, thrust_part, thrust_part])
            # 6 motores str([1.0, thrust_part, thrust_part, thrust_part, thrust_part, thrust_part, thrust_part])

                #Save do input modificado

                with open("./data/Input_CaravanDEP.xml", "w", encoding="utf-8") as f:
                    f.write(str(soup))

                def test_retrofit_DEP(rho):
                    logging.basicConfig(level=logging.WARNING)
                    logging.getLogger("fastoad.module_management._bundle_loader").disabled = True
                    logging.getLogger("fastoad.openmdao.variables.variable").disabled = True

                    # Define used files depending on options
                    #MODIFICAR COM O NUMERO DE MOTORES
                    xml_file_name = "input_CaravanDEP.xml"
                    process_file_name = "CaravanDEP_retrofit.yml" #nao muda

                    configurator = oad.FASTOADProblemConfigurator(pth.join(DATA_FOLDER_PATH, process_file_name))
                    problem = configurator.get_problem()

                    # Create inputs
                    ref_inputs = pth.join(DATA_FOLDER_PATH, xml_file_name)

                    problem.model_options["*propeller_1*"] = {"mass_as_input": True}
                    problem.model_options["*"] = {"cell_weight_ref": 50.0e-3 * 261.0 / rho}

                    problem.write_needed_inputs(ref_inputs)
                    problem.read_inputs()
                    problem.setup()
                
                    # om.n2(problem)

                    problem.run_model()

                    _, _, residuals = problem.model.get_nonlinear_vectors()
                    residuals = filter_residuals(residuals)

                    problem.write_outputs()

                    # deltaT = problem.get_val("performances.solve_equilibrium.compute_dep_equilibrium.compute_energy_consumed.power_train_performances.harness_1.temperature_profile.temperature.temperature_increase.cable_temperature_increase", units="K")
                    # print(deltaT)

                def test_CaravanDEP_mass_breakdown():
                    #MODIFICAR COM O NUMERO DE MOTORES
                    path_to_result_file = pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_CaravanDEP_retrofit.xml") 
                    path_to_pt_file = pth.join(DATA_FOLDER_PATH, "CaravanDEP_powertrain_retrofit.yml")

                    fig = power_train_mass_breakdown(path_to_result_file, path_to_pt_file)
                    fig.update_layout(uniformtext=dict(minsize=28))
                    fig.update_traces(textfont=dict(family=["Arial Black", "Arial"], size=[30]))
                    fig.show()
                    fig = api_plots.mass_breakdown_bar_plot(path_to_result_file)
                    fig.show()


                test_retrofit_DEP(rho)
                test_CaravanDEP_mass_breakdown()

                #MODIFICAR COM O NUMERO DE MOTORES
                filepath = pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_CaravanDEP_retrofit_2.xml") #"./results/oad_process_outputs_CaravanDEP_retrofit.xml"

                # # For reading the file content COMENTAR ISSO TUDO
                # file = open(filepath, 'r')
                # content = file.read()


                # For extracting a specific data from the .xml file
                tree = et.parse(filepath)
                root = tree.getroot()

                findData = root.find('./data')

                findTLAR = root.find('./data/TLAR')

                dist = float(findTLAR.find('range').text)/1852.0

                #Achar massas OWE, payload
                findAircraft = findData.find('./weight/aircraft')
                OWE = float(findAircraft.find('OWE').text) 

                payload = float(findAircraft.find('payload').text)

                #Flag para continuidade do Ciclo. Impor limite de HE para um avião com Certo motor elétrico e densidade de carga de bateria numa dada missão(alcance)

                if payload < 0:
                    for key in datum:
                        datum[key].append("-")
                    break
                
                #Continuidade do ciclo pra HE
#               #Obtenção de fuel_mass
                findsizing = findData.find('./mission/sizing')
                fuel_mass = float(findsizing.find('fuel').text)

#               #Emission Factor e total_emission
                findEnv_impact = findData.find('./environmental_impact/sizing')

                emission_factor = float(findEnv_impact.find('emission_factor').text)

                total_emissions = float(findEnv_impact.find('emissions').text)/1000

                #Empty Mass(Ultimo teste foi aqui)
                findAircraft_empty = findData.find('./weight/aircraft_empty')
                empty_mass = float(findAircraft_empty.find('mass').text)

                #He total mass e DC_dc_mass

                findHe = findData.find('./propulsion/he_power_train')
                he_mass = float(findHe.find('mass').text)

                findDC_DC = findHe.find('./DC_DC_converter/dc_dc_converter_1')
                dc_dc_mass = float(findDC_DC.find('mass').text)

                #Find battery mass

                findbat_mass = findHe.find('./battery_pack/battery_pack_1')

                battery_mass = float(findbat_mass.find('mass').text)

                #Fuel flowed

                findFuel_flw = findHe.find('./fuel_system/fuel_system_1')
                fuel_flowed = float(findFuel_flw.find('total_fuel_flowed').text)

                #Turboshaft

                findTurbo = findHe.find('./turboshaft/turboshaft_1')
                turboshaft_mass = float(findTurbo.find('mass').text)

                #Partes de somatorio

                #DC_bus somme

                findDC_bus = findHe.find('./DC_bus')

                bus_nose_mass = float(findDC_bus.find('./bus_nose/mass').text)

                #Busca pelos outros 6 elementos
                s = 0
                for i in range(n):
                    i_str = str(i+1)
                    finddc_bus_i = findDC_bus.find('./'+'dc_bus_'+ i_str)
                    s = s + float(finddc_bus_i.find('mass').text)
                dc_bus_mass = s + bus_nose_mass

                #Dc_cable_harness

                findDC_cable_harness = findHe.find('./DC_cable_harness')

                #Busca pelos outros 6 elementos
                harness_total = 0
                for i in range(n):
                    i_str = str(i+1)
                    finddc_harness_i = findDC_cable_harness.find('./'+'harness_'+ i_str)
                    harness_total = harness_total + float(finddc_harness_i.find('mass').text) + float(finddc_harness_i.find('./contactor/mass').text)

                #Motor mass

                findPMSM = findHe.find('./PMSM')

                #Busca pelos outros 6 elementos
                motor_total = 0
                for i in range(n):
                    i_str = str(i+1)
                    findmotor_i = findPMSM.find('./'+'motor_'+ i_str)
                    motor_total = motor_total + float(findmotor_i.find('mass').text)

                #Inverter mass

                findinv = findHe.find('./inverter')

                #Busca pelos outros 4 elementos (pq??)
                inverter_total = 0
                for i in range(1,n-1):
                    i_str = str(i+1)
                    findinv_i = findinv.find('./'+'inverter_'+ i_str)
                    inverter_total = inverter_total + float(findinv_i.find('mass').text)

                datum["Engine"].append("-")
                datum["rho_bat"].append(rho)
                datum["range_nmi"].append(dist)
                datum["he"].append(he)
                datum["fuel_mass"].append(fuel_mass)
                datum["emission_factor"].append(emission_factor)
                datum["total_emissions"].append(total_emissions)
                datum["red_emission"].append(100*(datum["total_emissions"][-1] - datum["total_emissions"][0])/datum["total_emissions"][0])
                datum["OWE"].append(OWE)
                datum["payload"].append(payload)
                datum["empty_mass"].append(empty_mass)
                datum["he_mass"].append(he_mass)
                datum["dc_dc_mass"].append(dc_dc_mass)
                datum["dc_bus_mass"].append(dc_bus_mass)
                datum["dc_cable_harn"].append(harness_total)
                datum["motor_mass"].append(motor_total)
                datum["battery_mass"].append(battery_mass)
                datum["fuel_flowed"].append(fuel_flowed)
                datum["inverter_mass"].append(inverter_total)
                datum["turboshaft_mass"].append(turboshaft_mass)
        
            for key in datum:
                datum[key].append("-")

            df_range = pd.DataFrame(datum)
            df = pd.concat([df,df_range])

df.to_csv('Adson_2.csv', index=True)





##################################################
#------------------------------------------------#
# Codigo feito pelo Felipe para extração embaixo #
#------------------------------------------------#
##################################################

            # from pathlib import Path
            # import sys
            # import xml.etree.ElementTree as ET
            # import re

            # path_to_result_file = pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_CaravanDEP_retrofit.xml")
            # # ------------------------------------------------------------
            # # Obtém caminho do XML (usa path_to_result_file se definido)
            # # ------------------------------------------------------------
            # try:
            #     xml_path = Path(path_to_result_file)  # pode lançar NameError se variável não existir
            # except NameError:
            #     if len(sys.argv) > 1:
            #         xml_path = Path(sys.argv[1])
            #     else:
            #         print("Erro: defina 'path_to_result_file' no ambiente ou passe o caminho como argumento.")
            #         sys.exit(1)

            # if not xml_path.exists():
            #     print(f"Erro: arquivo não encontrado: {xml_path}")
            #     sys.exit(2)

            # # ------------------------------------------------------------
            # # Carrega XML
            # # ------------------------------------------------------------
            # try:
            #     tree = ET.parse(str(xml_path))
            #     root = tree.getroot()
            # except ET.ParseError as e:
            #     print("Erro ao parsear XML:", e)
            #     sys.exit(3)

            # # ------------------------------------------------------------
            # # Localiza o nó <FASTOAD_model><data>
            # # ------------------------------------------------------------
            # def find_data_node(root):
            #     dn = root.find('.//FASTOAD_model/data')
            #     if dn is not None:
            #         return dn
            #     fm = root.find('.//FASTOAD_model')
            #     if fm is not None:
            #         dn = fm.find('.//data') or fm.find('data')
            #         if dn is not None:
            #             return dn
            #     all_data = root.findall('.//data')
            #     if all_data:
            #         return all_data[0]
            #     return None

            # data_node = find_data_node(root)
            # if data_node is None:
            #     print("Não foi possível localizar o nó <FASTOAD_model><data> no XML.")
            #     sys.exit(4)

            # # ------------------------------------------------------------
            # # Utilitários
            # # ------------------------------------------------------------
            # num_re = re.compile(r'[-+]?\d+(?:[.,]\d+)?(?:[eE][-+]?\d+)?')

            # def extract_numbers_from_text(s):
            #     if s is None:
            #         return []
            #     s = s.strip()
            #     if not s:
            #         return []
            #     found = num_re.findall(s)
            #     nums = []
            #     for tok in found:
            #         tok_clean = tok.replace(',', '.')
            #         try:
            #             nums.append(float(tok_clean))
            #         except ValueError:
            #             continue
            #     return nums

            # def text_of(elem):
            #     if elem is None:
            #         return ""
            #     return (elem.text or "").strip()

            # def try_find(node, path):
            #     if node is None:
            #         return None
            #     found = node.find(path)
            #     if found is not None:
            #         return found
            #     try:
            #         found = node.find('.//' + path)
            #         return found
            #     except Exception:
            #         return None

            # def get_first_text_by_path(path):
            #     el = try_find(data_node, path)
            #     if el is not None:
            #         return text_of(el)
            #     last = path.split('/')[-1]
            #     el = data_node.find('.//' + last)
            #     if el is not None:
            #         return text_of(el)
            #     return ""

            # def sum_masses_under(path_to_branch):
            #     """
            #     Soma todos os <mass> dentro do ramo path_to_branch (comportamento antigo).
            #     Mantido para outros usos.
            #     """
            #     branch = try_find(data_node, path_to_branch)
            #     if branch is None:
            #         last = path_to_branch.split('/')[-1]
            #         branch = data_node.find('.//' + last)
            #         if branch is None:
            #             return ""
            #     masses = branch.findall('.//mass')
            #     nums = []
            #     for m in masses:
            #         txt = text_of(m)
            #         nums.extend(extract_numbers_from_text(txt))
            #     if nums:
            #         total = sum(nums)
            #         if abs(total - round(total)) < 1e-9:
            #             return str(int(round(total)))
            #         else:
            #             return str(total)
            #     else:
            #         return ""

            # def sum_inverter_child_masses(path_to_inverter_branch, allowed_indices=(2,3,4,5)):
            #     """
            #     NOVA: soma apenas o <mass> que seja filho DIRETO de inverter_i (i in allowed_indices).
            #     Não desce em subcomponentes (capacitor, contactor, casing, ...).
            #     Retorna string vazia se não encontrar nada.
            #     """
            #     branch = try_find(data_node, path_to_inverter_branch)
            #     if branch is None:
            #         # tentativa por nome final
            #         last = path_to_inverter_branch.split('/')[-1]
            #         branch = data_node.find('.//' + last)
            #         if branch is None:
            #             return ""
            #     total_vals = []
            #     # iterar apenas sobre filhos imediatos
            #     for child in list(branch):
            #         tag = child.tag or ""
            #         # regex para inverter_i com underscore (ex: inverter_2) ou inverter2 (opcional)
            #         m = re.match(r'^inverter[_]?(\d+)$', tag, flags=re.IGNORECASE)
            #         if m:
            #             try:
            #                 idx = int(m.group(1))
            #             except ValueError:
            #                 continue
            #             if idx in allowed_indices:
            #                 # pegar mass filho direto (não descendente)
            #                 mass_node = child.find('mass')
            #                 if mass_node is not None:
            #                     nums = extract_numbers_from_text(text_of(mass_node))
            #                     if nums:
            #                         total_vals.append(nums[0])
            #                 # se inverter_i tiver atributo mass diretamente (caso texto no nó), também considerar
            #                 elif text_of(child):
            #                     nums = extract_numbers_from_text(text_of(child))
            #                     if nums:
            #                         total_vals.append(nums[0])
            #         # também lidar com caso extraordinário: filho chamado exatamente 'inverter' que contenha inverter_i dentro
            #         # mas não queremos descer abaixo de inverter_i, então ignoramos aqui.
            #     if total_vals:
            #         total = sum(total_vals)
            #         if abs(total - round(total)) < 1e-9:
            #             return str(int(round(total)))
            #         else:
            #             return str(total)
            #     return ""

            # # ------------------------------------------------------------
            # # Mapeamento dos campos (nome de saída -> caminho relativo)
            # # ------------------------------------------------------------
            # fields_map = [
            #     ("fuel", "mission/sizing/fuel"),
            #     ("emission_factor", "environmental_impacts/sizing/emission_factor"),
            #     ("emissions", "environmental_impacts/sizing/emissions"),
            #     ("OWE", "weight/aircraft/OWE"),
            #     ("payload", "weight/aircraft/payload"),
            #     ("mass_aircraft_empty", "weight/aircraft_empty/mass"),
            #     ("mass_he_power_train", "propulsion/he_power_train/mass"),
            #     ("mass_dc_dc_converter_1", "propulsion/he_power_train/DC_DC_converter/dc_dc_converter_1/mass"),
            #     ("mass_dc_bus_total", "propulsion/he_power_train/DC_bus"),
            #     ("mass_dc_cable_harness_total", "propulsion/he_power_train/DC_cable_harness"),
            #     ("mass_pmsm_total", "propulsion/he_power_train/PMSM"),
            #     ("mass_battery_pack_1", "propulsion/he_power_train/battery_pack/battery_pack_1/mass"),
            #     ("total_fuel_flowed", "propulsion/he_power_train/fuel_system/fuel_system_1/total_fuel_flowed"),
            #     # inverter: usar nova função que limita a descida ao ramo inverter_i (2..5)
            #     ("mass_inverter_total", "propulsion/he_power_train/inverter"),
            #     ("mass_turboshaft_1", "propulsion/turboshaft/turboshaft_1/mass"),
            # ]

            # # ------------------------------------------------------------
            # # Coleta valores
            # # ------------------------------------------------------------
            # headers = []
            # values = []
            # for name, path in fields_map:
            #     headers.append(name)
            #     if name in ("mass_dc_dc_converter_1", "mass_battery_pack_1", "mass_turboshaft_1"):
            #         txt = get_first_text_by_path(path)
            #         nums = extract_numbers_from_text(txt)
            #         if nums:
            #             val = nums[0]
            #             values.append(str(int(round(val))) if abs(val - round(val)) < 1e-9 else str(val))
            #         else:
            #             values.append("")
            #     elif name in ("mass_dc_bus_total", "mass_dc_cable_harness_total", "mass_pmsm_total"):
            #         summed = sum_masses_under(path)
            #         values.append(summed)
            #     elif name == "total_fuel_flowed":
            #         txt = get_first_text_by_path(path)
            #         nums = extract_numbers_from_text(txt)
            #         if nums:
            #             val = nums[0] if len(nums) == 1 else sum(nums)
            #             values.append(str(int(round(val))) if abs(val - round(val)) < 1e-9 else str(val))
            #         else:
            #             values.append(txt)
            #     elif name == "mass_inverter_total":
            #         # usa a função especializada que só desce até inverter_i (2..5)
            #         summed = sum_inverter_child_masses(path_to_inverter_branch=path, allowed_indices=(2,3,4,5))
            #         values.append(summed)
            #     else:
            #         txt = get_first_text_by_path(path)
            #         nums = extract_numbers_from_text(txt)
            #         if nums and (txt.strip() == str(nums[0]) or any(c.isdigit() for c in txt)):
            #             val = nums[0]
            #             values.append(str(int(round(val))) if abs(val - round(val)) < 1e-9 else str(val))
            #         else:
            #             values.append(txt)

            # # ------------------------------------------------------------
            # # Saída (tab-separated)
            # # ------------------------------------------------------------
            # sep = "\t"
            # print(sep.join(headers))
            # print(sep.join(values))


Old value: 400.0 nmi


c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\openmdao\core\component.py:1189: OMDeprecationWarning:'performances.solve_equilibrium.compute_dep_equilibrium.compute_energy_consumed.power_train_performances.turboshaft_1.density_ratio' <class PerformancesDensityRatio>: d(density_ratio)/d(rpm): Partial was declared to be exactly zero. This is inefficient and the declaration should be removed. In a future version of OpenMDAO this behavior will raise an error.
c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\openmdao\core\component.py:1189: OMDeprecationWarning:'power_train_sizing.battery_pack_1.battery_drag_ls' <class SizingBatteryDrag>: d(*)/d(*): Partial was declared to be exactly zero. This is inefficient and the declaration should be removed. In a future version of OpenMDAO this behavior will raise an error.
c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\openmdao\core\component.py:1189: OMDeprecationWarning:'power_train_sizing.battery_pack_1.b

NL: NLBGS 1 ; 3.56405105e+11 1


C:\FAST-OAD-CS23-HE-main\src\fastga_he\models\propulsion\components\propulsor\propeller\components\perf_efficiency.py:53: RuntimeWarning:

divide by zero encountered in divide

c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\openmdao\solvers\solver.py:782: SolverWarning:Your model has stalled three times and may be violating the bounds. In the future, turn on print_bound_enforce in your solver options here: 
performances.solve_equilibrium.compute_dep_equilibrium.nonlinear_solver.linesearch.options['print_bound_enforce']=True. 
The bound(s) being violated now are:

c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\openmdao\solvers\linesearch\backtracking.py:35: SolverWarning:'performances.solve_equilibrium.compute_dep_equilibrium.compute_energy_consumed.power_train_performances.inverter_1.modulation_idx.modulation_index' exceeds upper bounds
  Val: [3.29840389 3.29840949 3.29840957 ... 3.29820609 3.29820606 3.29820605]
  Upper: [2. 2. 2. ... 2. 2. 2.]

c:\Users

|  |  NL: NewtonSolver 'NL: Newton' on system 'performances.solve_equilibrium.compute_dep_equilibrium' stalled after 8 iterations.


...

C:\FAST-OAD-CS23-HE-main\src\fastga_he\models\propulsion\components\propulsor\propeller\components\perf_efficiency.py:53: RuntimeWarning:

divide by zero encountered in divide

c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\openmdao\solvers\solver.py:782: SolverWarning:Your model has stalled three times and may be violating the bounds. In the future, turn on print_bound_enforce in your solver options here: 
performances.solve_equilibrium.compute_dep_equilibrium.nonlinear_solver.linesearch.options['print_bound_enforce']=True. 
The bound(s) being violated now are:

c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\openmdao\solvers\linesearch\backtracking.py:35: SolverWarning:'performances.solve_equilibrium.compute_dep_equilibrium.compute_energy_consumed.power_train_performances.battery_pack_1.efficiency.efficiency' exceeds upper bounds
  Val: [0.99064998 0.99067945 0.99068373 ... 0.98952348 0.98791353 0.98619229]
  Upper: [1. 1. 1. ... 1. 1. 1.]

c:\Users\joaor

|  |  NL: NewtonSolver 'NL: Newton' on system 'performances.solve_equilibrium.compute_dep_equilibrium' stalled after 7 iterations.
|  NL: NLBGSSolver 'NL: NLBGS' on system 'performances' stalled after 3 iterations.
NL: NLBGS 12 ; 2512786.84 7.05036714e-06
NL: NLBGS Converged
[ 5.38409622e-01  7.55738567e-03 -1.40049586e-02 ... -3.06603094e-03
 -1.16905308e-03  1.36621482e+01]


c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\fastoad\_utils\strings.py:55: UserWarning:

genfromtxt: Empty input file: "<_io.StringIO object at 0x000001F33EE5BAC0>"

c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\fastoad\_utils\strings.py:55: UserWarning:

genfromtxt: Empty input file: "<_io.StringIO object at 0x000001F33EE59870>"

c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\fastoad\_utils\strings.py:55: UserWarning:

genfromtxt: Empty input file: "<_io.StringIO object at 0x000001F33EE581F0>"



Difference found between power train weight and the sum of the components, 1644.785261 vs 1644.785263


c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\fastoad\_utils\strings.py:55: UserWarning:

genfromtxt: Empty input file: "<_io.StringIO object at 0x000001F33EE5A200>"

c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\fastoad\_utils\strings.py:55: UserWarning:

genfromtxt: Empty input file: "<_io.StringIO object at 0x000001F33EE58670>"

c:\Users\joaor\miniconda3\envs\FAST-OAD-ENV\lib\site-packages\fastoad\_utils\strings.py:55: UserWarning:

genfromtxt: Empty input file: "<_io.StringIO object at 0x000001F33EE58F70>"



In [ ]:
import os.path as pth
from shutil import rmtree
import logging

import pytest

import openmdao.api as om
import fastoad.api as oad
import fastga.utils.postprocessing.analysis_and_plots as api_plots

from fastga.utils.postprocessing.analysis_and_plots import (
    mass_breakdown_bar_plot,
    aircraft_geometry_plot,
)

fig = api_plots.aircraft_geometry_plot(pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_CaravanDEP_retrofit.xml"), name="")
fig = api_plots.aircraft_geometry_plot(pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_elec_CaravanDEP_retrofit.xml"), name="", fig = fig)
fig = api_plots.aircraft_geometry_plot(pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_elec2_CaravanDEP_retrofit.xml"), name="", fig=fig)
fig = api_plots.aircraft_geometry_plot(pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_elec3_CaravanDEP_retrofit.xml"), name="", fig=fig)
fig.show()

In [ ]:
import os.path as pth
import logging
import shutil


import fastoad.api as oad
import fastga_he.api as oad_he

from fastga_he.gui.power_train_network_viewer import power_train_network_viewer
from fastga_he.gui.analysis_and_plots import (
    aircraft_geometry_plot,
)
from fastga_he.gui.power_train_weight_breakdown import (
    power_train_mass_breakdown,
)
from fastga_he.gui.payload_range import (
    payload_range_outer,
)
DATA_FOLDER_PATH = "data"

WORK_FOLDER_PATH = "workdir"

CONFIGURATION_FILE = pth.join(DATA_FOLDER_PATH, "CaravanDEP_retrofit.yml")
#PT_FILE = pth.join(WORK_FOLDER_PATH, "simple_assembly.yml")
SOURCE_FILE = pth.join(DATA_FOLDER_PATH, "input_CaravanDEP.xml")
RESULTS_FOLDER_PATH = "results"

# For having log messages on screen
logging.basicConfig(level=logging.WARNING, format="%(levelname)-8s: %(message)s")

In [ ]:
# We copy all the useful file in the workdir

shutil.copy(pth.join(DATA_FOLDER_PATH, "mission_vector.yml"), CONFIGURATION_FILE)
shutil.copy(pth.join(DATA_FOLDER_PATH, "simple_assembly.yml"), PT_FILE)

In [ ]:
pt_file_path = pth.join(DATA_FOLDER_PATH, "CaravanDEP_powertrain_retrofit_2.yml")
network_file_path = pth.join(RESULTS_FOLDER_PATH, "CaravanDEP_powertrain_retrofit_2.html")
power_train_network_viewer(pt_file_path, network_file_path)

In [ ]:
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)

In [ ]:
oad.variable_viewer(oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True))

In [ ]:
from fastoad import api as api_cs25
from IPython.display import IFrame

N2_FILE = pth.join(DATA_FOLDER_PATH, "n2.html")
api_cs25.write_n2(CONFIGURATION_FILE, N2_FILE, overwrite=True)

IFrame(src=N2_FILE, width="100%", height="500px")

Now we run the problem

In [ ]:
eval_problem = oad.evaluate_problem(CONFIGURATION_FILE, overwrite=True)

In [ ]:
pt_file_path = pth.join(DATA_FOLDER_PATH, "CaravanDEP_powertrain_retrofit.yml")
power_train_mass_breakdown(eval_problem.output_file_path, pt_file_path)

In [ ]:
# find_bad_vars.py
import re, os, sys
import numpy as np

FNAME = "solver_errors.0.out"
if not os.path.exists(FNAME):
    # tenta achar o mais recente
    import glob
    cands = sorted(glob.glob("solver_errors*.out"), key=os.path.getmtime)
    if cands:
        FNAME = cands[-1]
    else:
        print("Arquivo solver_errors*.out não encontrado no diretório:", os.getcwd())
        sys.exit(1)

text = open(FNAME, "r", encoding="utf-8", errors="replace").read()
print("Usando arquivo:", FNAME)

# regex encontra 'key': array(
pattern = re.compile(r"(?P<quote>['\"])(?P<key>.+?)(?P=quote)\s*:\s*array\s*\(", flags=re.MULTILINE)

bad = []
found_specific = {"efficiency": [], "shaft_power_out": [], "power": []}
total = 0

for m in pattern.finditer(text):
    total += 1
    key = m.group("key")
    start = m.end()
    # achar fechamento correspondente de parênteses
    depth = 1
    i = start
    N = len(text)
    while i < N and depth > 0:
        ch = text[i]
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        i += 1
    arr_block = text[m.start():i]
    arr_text = arr_block.split(":",1)[1].strip()
    # preparar para eval: transformar array( -> np.array( e nan/inf
    arr_text2 = arr_text.replace("array(", "np.array(")
    arr_text2 = re.sub(r"\bnan\b", "np.nan", arr_text2, flags=re.IGNORECASE)
    arr_text2 = re.sub(r"\binf\b", "np.inf", arr_text2, flags=re.IGNORECASE)
    try:
        val = eval(arr_text2, {"np": np})
    except Exception:
        # fallback: extrair números/np.nan/np.inf
        inner = re.sub(r'^\s*np\.array\s*\(', '', arr_text2)
        if inner.endswith(')'):
            inner = inner[:-1]
        tokens = re.findall(r"np\.nan|np\.inf|[-+]?\d*\.\d+|\d+|[-+]?\d+\.\d+[eE][-+]?\d+|[-+]?\d+[eE][-+]?\d+", inner)
        vals = []
        for t in tokens:
            if t.lower() == "np.nan":
                vals.append(np.nan)
            elif t.lower() == "np.inf":
                vals.append(np.inf)
            else:
                try:
                    vals.append(float(t))
                except:
                    pass
        val = np.array(vals) if vals else None

    if val is None:
        continue
    arr = np.asarray(val)
    if arr.size == 0:
        continue
    mask = ~np.isfinite(arr)
    if np.any(mask):
        idx = np.where(mask)[0][:30].tolist()
        sample = arr[idx].tolist()
        bad.append((key, arr.shape, len(idx), idx[:10], sample[:10]))
    # armazena casos com nomes sugestivos
    lname = key.lower()
    if "efficiency" in lname or "efficiency" in key:
        found_specific["efficiency"].append((key, arr.shape, np.isnan(arr).sum(), np.isinf(arr).sum()))
    if "shaft_power_out" in lname or "shaft_power_out" in key:
        found_specific["shaft_power_out"].append((key, arr.shape, np.isnan(arr).sum(), np.isinf(arr).sum()))
    if "power" in lname and "active" in lname or "power" in lname:
        found_specific["power"].append((key, arr.shape, np.isnan(arr).sum(), np.isinf(arr).sum()))

# imprime resumo
print(f"Total de entradas analisadas (padrão 'array('): {total}")
print("Encontradas variáveis com NaN/Inf:", len(bad))
if len(bad) > 0:
    print("\n== Top 60 variáveis com não-finitos (nome | shape | n_nonfinite | índices sample | valores sample):")
    for item in bad[:60]:
        print(item)

print("\n== Variáveis relacionadas (efficiency / shaft_power_out / power) encontradas:")
for k,v in found_specific.items():
    if v:
        print(f"\n{k}:")
        for t in v:
            print("  ", t)
    else:
        print(f"\n{k}: Nenhuma encontrada")

# salvar lista curta
with open("bad_vars_quick.txt", "w", encoding="utf-8") as f:
    for item in bad:
        f.write(f"{item}\n")
print("\nRelatório salvo em bad_vars_quick.txt")


In [ ]:
from openmdao.core.problem import Problem as OMP

_orig_setup = OMP.setup

def _patched_setup(self, *args, **kwargs):
    ret = _orig_setup(self, *args, **kwargs)
    # tenta setar as opções de debug assim que o model existir
    try:
        if hasattr(self, 'model') and self.model is not None:
            # opções comuns
            try:
                self.model.nonlinear_solver.options['iprint'] = 2
                self.model.nonlinear_solver.options['debug_print'] = True
                self.model.nonlinear_solver.options['err_on_non_converge'] = True
            except Exception:
                # se o sistema não tiver nonlinear_solver padrão, percorre subsistemas
                for s in self.model.system_iter(include_self=True, recurse=True):
                    try:
                        s.nonlinear_solver.options['iprint'] = 2
                        s.nonlinear_solver.options['debug_print'] = True
                    except Exception:
                        pass
            # print detalhado de solver
            try:
                self.model.set_solver_print(level=2, depth=3, type_='NL')
            except Exception:
                pass
    except Exception:
        pass
    return ret

# aplica o monkeypatch
OMP.setup = _patched_setup

# agora chama sua função normalmente — quando o Problem for criado + setup(), as opções serão aplicadas
eval_problem = oad.evaluate_problem(CONFIGURATION_FILE, overwrite=True)

In [ ]:

# inspect_solver_out.py
import os
import re
import sys
import numpy as np
from pprint import pprint

FNAME = "solver_errors.0.out"

if not os.path.exists(FNAME):
    print("Arquivo não encontrado:", FNAME)
    sys.exit(1)

# lê o arquivo como texto (substitui bytes inválidos)
with open(FNAME, "r", encoding="utf-8", errors="replace") as f:
    text = f.read()

# ------------------------------------------------------------
# 1) tentativa direta: transformar 'array(' -> 'np.array(' e normalizar nan/inf
# ------------------------------------------------------------
transformed = text

# substitui array( -> np.array(
transformed = transformed.replace("array(", "np.array(")

# substitui 'nan' isolado por 'np.nan', sem tocar em 'np.nan' já existentes
transformed = re.sub(r'(?<!np\.)\bnan\b', 'np.nan', transformed, flags=re.IGNORECASE)

# substitui 'inf' isolado por 'np.inf', sem tocar em 'np.inf' já existentes
transformed = re.sub(r'(?<!np\.)\binf\b', 'np.inf', transformed, flags=re.IGNORECASE)

# tenta avaliar o dict inteiro (pode falhar em arquivos com comentários ou sintaxe extra)
obj = None
ok = False
print("Tentando eval do arquivo transformado (pode demorar um pouco)...")
try:
    obj = eval(transformed, {"np": np})
    if isinstance(obj, dict):
        print("Eval completo funcionou: arquivo convertido em dict com", len(obj), "chaves top-level.")
        ok = True
    else:
        print("Eval devolveu tipo:", type(obj))
except Exception as e:
    print("Eval completo falhou:", e)

# ------------------------------------------------------------
# 2) fallback: extrair pares 'nome': array(...) com parsing tolerante
# ------------------------------------------------------------
if not ok:
    print("Fazendo fallback: extraindo pares 'nome': array(...) com regex/parsing tolerante.")
    extracted = {}

    # regex que encontra ocorrências de 'key': array( (captura a posição da chave)
    pattern = re.compile(r"(?P<quote>['\"])(?P<key>.+?)(?P=quote)\s*:\s*array\s*\(", flags=re.MULTILINE)

    for m in pattern.finditer(text):
        key = m.group("key")
        start = m.end()  # posição logo após "array("
        # varrer para encontrar o parêntese de fechamento correspondente
        depth = 1
        i = start
        N = len(text)
        # percorre caractere a caractere até fechar todos os parênteses do array(...)
        while i < N and depth > 0:
            ch = text[i]
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
            i += 1
        arr_block = text[m.start():i]  # bloco contendo "'key': array(...)" (ou parte)
        # extrai apenas a parte depois dos dois-pontos
        try:
            arr_text_eval = arr_block.split(":", 1)[1].strip()
        except Exception:
            arr_text_eval = arr_block

        # transforma array( -> np.array( e normaliza nan/inf (usando formas seguras)
        arr_text_eval = arr_text_eval.replace("array(", "np.array(")
        arr_text_eval = re.sub(r'(?<!np\.)\bnan\b', 'np.nan', arr_text_eval, flags=re.IGNORECASE)
        arr_text_eval = re.sub(r'(?<!np\.)\binf\b', 'np.inf', arr_text_eval, flags=re.IGNORECASE)

        # tenta eval seguro do array (namespace limitado)
        val = None
        try:
            val = eval(arr_text_eval, {"np": np})
        except Exception:
            # tentativa de fallback simplificado: extrair números e np.nan/np.inf por regex
            inner = arr_text_eval
            # remover "np.array(" prefix e parênteses finais se existirem
            inner = re.sub(r'^\s*np\.array\s*\(', '', inner)
            if inner.endswith(')'):
                inner = inner[:-1]
            # agora procurar tokens numéricos e np.nan / np.inf
            tokens = re.findall(r"np\.nan|np\.inf|[-+]?\d*\.\d+|\d+|[-+]?\d+\.\d+[eE][-+]?\d+|[-+]?\d+[eE][-+]?\d+", inner)
            vals = []
            for t in tokens:
                if t.lower() == "np.nan":
                    vals.append(np.nan)
                elif t.lower() == "np.inf":
                    vals.append(np.inf)
                else:
                    try:
                        vals.append(float(t))
                    except Exception:
                        # ignora tokens não convertíveis
                        pass
            val = np.array(vals) if vals else None

        extracted[key] = val

    obj = extracted
    print("Fallback extraiu", len(extracted), "entradas.")

# ------------------------------------------------------------
# 3) analisar o dict/extraído e gerar relatório com variáveis que têm NaN/Inf
# ------------------------------------------------------------
def analyze_dict(D):
    bad = []
    if not isinstance(D, dict):
        return bad
    for k, v in D.items():
        try:
            arr = np.asarray(v)
            if arr.size > 0:
                mask = ~np.isfinite(arr)
                if np.any(mask):
                    idx = np.where(mask)[0][:20].tolist()
                    sample_vals = arr[idx].tolist()
                    bad.append((k, arr.shape, idx, sample_vals))
        except Exception:
            # tenta avaliar se é um scalar não-finite
            try:
                fv = float(v)
                if not np.isfinite(fv):
                    bad.append((k, (), [], fv))
            except Exception:
                pass
    return bad

bad_list = analyze_dict(obj)

if bad_list:
    print("\n== Variáveis com NaN/Inf detectadas:", len(bad_list))
    with open("bad_vars.txt", "w", encoding="utf-8") as fout:
        for name, shape, idx_list, sample_vals in bad_list:
            line = f"{name} | shape={shape} | indices_sample={idx_list} | sample_bad_values={sample_vals}\n"
            fout.write(line)
            print(line.strip())
    print("\nRelatório salvo como bad_vars.txt")
else:
    print("Nenhuma variável com NaN/Inf detectada (ou nada foi extraído).")

We can now use the API to create graphs based on the data saved during mission computation

In [ ]:
import os.path as pth

import fastoad.api as oad
import fastga.utils.postprocessing.post_processing_api as api_plots

# For using all screen width
#from IPython.core.display import display, HTML

display(HTML("<style>.container { width:95% !important; }</style>"))

fig = api_plots.mass_breakdown_sun_plot(eval_problem.output_file_path)
fig.show()

In [ ]:
RESULTS_FOLDER_PATH = "results"
MISSION_DATA_FILE = pth.join(RESULTS_FOLDER_PATH, "turbo_electric_propulsion.csv")
PT_DATA_FILE = pth.join(RESULTS_FOLDER_PATH, "turbo_electric_propulsion_pt_watcher.csv")

perfo_viewer = oad_he.PerformancesViewer(
    power_train_data_file_path=PT_DATA_FILE,
    mission_data_file_path=MISSION_DATA_FILE,
    plot_height=800,
)

# Uncomment next lines if you want raw data
# pd.set_option('display.max_rows', 500)
# pd.set_option('display.max_columns', 500)
# pd.set_option('display.width', 200)
# print(perfo_viewer.data)

In [ ]:
oad.variable_viewer(eval_problem.output_file_path)

Let us try optimizing the propeller to get better efficiencies.

In [ ]:
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)
oad.optimization_viewer(CONFIGURATION_FILE)

Uncomment next cell if you want to run an optimization, does not seems to work very well with Cobyla (even worse with SLSQP).

In [ ]:
# optim_problem = oad.optimize_problem(CONFIGURATION_FILE, overwrite=True)

In [ ]:
oad.optimization_viewer(CONFIGURATION_FILE)